# F5-TTS Vietnamese ViVoice — MN 1 smoke test

Research-only test. This notebook uses the private Kaggle Dataset checkpoint and the canonical MN 1 translation snapshot. The model is licensed `CC-BY-NC-SA-4.0` (non-commercial research use only); generated audio is AI-generated.

Default mode is a short smoke test. Set `RUN_MODE = 'full'` only after the smoke output is valid.

In [ ]:
import subprocess
import sys

# Kaggle's GPU image supplies CUDA/PyTorch; install only the upstream package and its declared dependencies.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/nguyenthienhy/F5-TTS-Vietnamese.git@e74db9d5a5e521bb930490e7cd7912438bf7ae84'], check=True)
print('F5-TTS package installed')

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
from urllib.request import urlopen
import torch

RUN_MODE = 'full'  # smoke validation passed; generate the complete MN 1 text
SPEED = 0.75
MODEL_ROOT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/f5-vivvoice-output')
TRANSLATION_URL = 'https://raw.githubusercontent.com/streamentry/kinh-tang-pali/1770865c0dba9f5caddbc901565450768f274c05/content/translation/vi/project/sutta/mn/mn1_translation-vi-project.json'
TRANSLATION = Path('/kaggle/working/mn1_translation_snapshot.json')
checkpoint = next(MODEL_ROOT.rglob('model_last.pt'))
vocab = checkpoint.with_name('vocab.txt')
reference_audio = checkpoint.with_name('ref.wav')
if not TRANSLATION.exists():
    TRANSLATION.write_bytes(urlopen(TRANSLATION_URL, timeout=60).read())
translation = json.loads(TRANSLATION.read_text(encoding='utf-8'))
values = list(translation.values())
if RUN_MODE == 'smoke':
    values = values[:12]
script_path = OUTPUT_DIR / f'mn1-{RUN_MODE}.txt'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
script_path.write_text('\n\n'.join(values), encoding='utf-8')
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
print('checkpoint:', checkpoint)
print('checkpoint_bytes:', checkpoint.stat().st_size)
print('mode:', RUN_MODE, 'speed:', SPEED)
print('segments:', len(values), 'characters:', sum(len(value) for value in values))

In [ ]:
output_name = f'mn1-f5-vivvoice-{RUN_MODE}.wav'
command = [
    sys.executable, '-m', 'f5_tts.infer.infer_cli',
    '--model', 'F5TTS_Base',
    '--ref_audio', str(reference_audio),
    '--ref_text', 'cả hai bên hãy cố gắng hiểu cho nhau',
    '--gen_file', str(script_path),
    '--speed', str(SPEED),
    '--vocoder_name', 'vocos',
    '--vocab_file', str(vocab),
    '--ckpt_file', str(checkpoint),
    '--output_dir', str(OUTPUT_DIR),
    '--output_file', output_name,
    '--remove_silence',
]
print('running:', ' '.join(command))
subprocess.run(command, check=True)
audio = OUTPUT_DIR / output_name
if not audio.is_file() or audio.stat().st_size == 0:
    raise RuntimeError(f'inference completed without output: {audio}')
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(16 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
manifest = OUTPUT_DIR / f'mn1-f5-vivvoice-{RUN_MODE}.manifest.json'
manifest.write_text(json.dumps({
    'model': 'hynt/F5-TTS-Vietnamese-ViVoice',
    'mode': RUN_MODE,
    'speed': SPEED,
    'reference_text': 'cả hai bên hãy cố gắng hiểu cho nhau',
    'translation_source_url': TRANSLATION_URL,
    'translation_segment_count': len(values),
    'translation_character_count': sum(len(value) for value in values),
    'checkpoint_sha256': sha256(checkpoint),
    'output_sha256': sha256(audio),
    'output_bytes': audio.stat().st_size,
}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('audio:', audio, audio.stat().st_size, 'bytes')
print('manifest:', manifest)

In [ ]:
import json
import soundfile as sf

samples, sample_rate = sf.read(audio)
details = json.loads(manifest.read_text(encoding='utf-8'))
print(json.dumps({
    'duration_seconds': round(len(samples) / sample_rate, 3),
    'sample_rate': sample_rate,
    'channels': 1 if samples.ndim == 1 else samples.shape[1],
    'output_sha256': details['output_sha256'],
    'checkpoint_sha256': details['checkpoint_sha256'],
}, ensure_ascii=False, indent=2))